# 10 -- OptionSuite & OptionMetrics Data Collection

## Purpose
Collects and assembles a comprehensive daily stock-level options factor panel by combining OptionSuite pre-computed factors with custom extractions from OptionMetrics raw tables on WRDS. The pipeline runs across six stages, each building on the previous, and produces a single merged panel keyed on `(permno, date)`.

## Source
- **OptionSuite:** Pre-downloaded CSV (`optionsuite_download.csv`) containing daily options-derived factors (IV, skew, put-call ratios, historical vol, open interest by moneyness bucket, etc.)
- **OptionMetrics on WRDS** (via the `wrds` Python library, authenticated with username `henrylavender`):
  - `optionm.secnmd` -- security name/CUSIP mapping
  - `optionm.vsurfd{YYYY}` -- standardised volatility surface (yearly tables, 2004--2024)
  - `optionm.opprcd{YYYY}` -- option prices and Greeks (yearly tables, 2004--2024)
  - `optionm.secprd` -- underlying security prices
  - `optionm.stdbrte{YYYY}` -- standardised implied borrow rates (yearly tables, 2004--2024)
- **CRSP** (`crsp.stocknames` for identifier linking, `crsp_a_stock.dsf_v2` for market cap and daily returns)

## Input
`Data/Data_Collection/Initial/05_Top100_SP500_Universe/universe_master.parquet` -- master PERMNO list from notebook 05.

---

## Stage 1: Filter OptionSuite Base Data

The raw OptionSuite CSV is filtered to only PERMNOs in the master universe list using DuckDB (avoids loading the full file into memory). The inner join is on `PERMNO`, and the result is saved directly to parquet.

**Output:** `Data/Data_Collection/Initial/10_OptionSuite/options_filtered.parquet`

---

## Stage 2: OptionMetrics to CRSP Identifier Crosswalk

Builds a date-aware mapping from OptionMetrics `secid` to CRSP `permno` by joining through 8-digit CUSIP.

### Join Logic
- `optionm.secnmd` maps `secid` to `cusip`, valid from `effect_date` onwards
- `crsp.stocknames` maps `ncusip` to `permno`, valid from `namedt` to `nameenddt`
- The link is valid where `GREATEST(effect_date, namedt) <= nameenddt`
- Filtered to master PERMNOs only, with `nameenddt >= '2004-01-01'`

### Output Columns
`secid`, `cusip`, `permno`, `link_start_date`, `link_end_date`

**Output:** `Data/Data_Collection/Initial/10_OptionSuite/om_crosswalk.parquet`

---

## Stage 3: Volatility Surface -- Term Structure & Smile

Extracts additional volatility surface factors from OptionMetrics that OptionSuite does not provide. OptionSuite already provides ATM call IV (~30d), ATM put IV, OTM put IV, and skew; this stage adds the term structure and smile dimensions.

### Table Queried
`optionm.vsurfd{YYYY}` (yearly tables, 2004--2024), filtered to the secid universe from the crosswalk.

### Collection Method
Conditional aggregation in SQL produces one row per `(secid, date)`. The query filters to `days IN (30, 91)` and `delta IN (25, 50, -25)` and extracts:
- `iv_30d_atm_vsurfd` -- 30-day ATM call IV (days=30, delta=50, cp_flag='C')
- `iv_91d_atm` -- 91-day ATM call IV (days=91, delta=50, cp_flag='C')
- `iv_30d_call25` -- 30-day 25-delta OTM call IV (days=30, delta=25, cp_flag='C')
- `iv_30d_put25` -- 30-day 25-delta OTM put IV (days=30, delta=-25, cp_flag='P')

### Derived Factors
- `vol_term_structure` = `iv_91d_atm` - `iv_30d_atm_vsurfd` (positive means upward-sloping term structure; typically goes negative in stress)
- `vol_smile` = `iv_30d_put25` + `iv_30d_call25` - 2 * `iv_30d_atm_vsurfd` (measures convexity/curvature of the smile; positive on average)

### Output
- Per-year: `Data/Data_Collection/Initial/10_OptionSuite/om_vol_surface_yearly/om_vol_surface_YYYY.parquet`
- Combined: `Data/Data_Collection/Initial/10_OptionSuite/om_vol_surface_all.parquet`

---

## Stage 4: Greeks & Positioning Factors

Computes daily stock-level Greeks-based positioning factors from OptionMetrics option price tables. All heavy computation runs on the WRDS Postgres server via SQL using a CTE.

### Tables Queried
- `optionm.opprcd{YYYY}` -- option prices, Greeks, open interest, volume (yearly tables, 2004--2024)
- `optionm.secprd` -- underlying stock close price (joined on secid and date)

### Filters
- `secid` restricted to the crosswalk universe
- `open_interest > 0`
- `impl_volatility IS NOT NULL`
- `delta IS NOT NULL AND delta BETWEEN -1 AND 1`
- `gamma IS NOT NULL`
- `stock close IS NOT NULL AND close > 0`

### Per-Contract Intermediates (CTE)
- `gex_contract` = gamma * open_interest * contract_size * stock_close^2 * 0.01
- `dex_contract` = delta * open_interest * contract_size * stock_close
- `toxicity_contract` = delta * volume * contract_size * stock_close (only when volume > 0)

### Aggregated Factors (per secid, date)
- `gex` -- Gamma Exposure: sum of call GEX minus sum of put GEX. Measures dealer hedging pressure.
- `dex` -- Delta Exposure: sum of all dex_contract. Net directional positioning.
- `delta_dollar_volume` -- Flow toxicity: sum of toxicity_contract. Delta-weighted dollar volume.
- `oi_wt_delta` -- Open-interest-weighted average delta (net positioning tilt)
- `oi_wt_gamma` -- Open-interest-weighted average gamma (convexity exposure)
- `oi_wt_vega` -- Open-interest-weighted average vega (vol sensitivity)
- `oi_wt_theta` -- Open-interest-weighted average theta (time decay exposure)
- `total_oi` -- Total open interest (diagnostic)
- `total_volume` -- Total volume (diagnostic)

### Output
- Per-year: `Data/Data_Collection/Initial/10_OptionSuite/om_greeks_positioning_yearly/om_greeks_positioning_YYYY.parquet`
- Combined: `Data/Data_Collection/Initial/10_OptionSuite/om_greeks_positioning_all.parquet`

---

## Stage 5: Volatility Risk Premium (VRP) & Implied Borrow Rates

### Part A: Volatility Risk Premium

**Inputs:**
- `options_filtered.parquet` (OptionSuite base, provides `iv_catm` and `hvol`)
- CRSP daily returns from `Data/Data_Collection/Initial/06_Daily_CRSP_Stock_Data/firm_daily` (for independent realised vol computation)

**Unit normalisation:** Both `iv_catm` and `hvol` are checked for scale (percentage vs decimal) and divided by 100 if the mean exceeds 1.0.

**Realised volatility computation:** `rv_30d` is computed as the trailing 30-trading-day standard deviation of returns (minimum 20 observations), annualised by multiplying by sqrt(252). Returns are shifted by one day (`shift(1)`) so that the calculation uses days t-1 through t-30, strictly avoiding look-ahead.

**VRP factors:**
- `vrp_hvol` = `iv_catm` - `hvol` (using OptionSuite's matched 30-day historical vol)
- `vrp_rv` = `iv_catm` - `rv_30d` (using independently computed CRSP realised vol)

Both should be positive on average (implied vol typically exceeds realised vol -- the variance risk premium).

**Output:** `Data/Data_Collection/Initial/10_OptionSuite/om_vrp.parquet` -- columns: `permno`, `date`, `iv_catm`, `rv_30d`, `hvol`, `vrp_rv`, `vrp_hvol`

### Part B: Implied Borrow Rates

**Table queried:** `optionm.stdbrte{YYYY}` (yearly tables, 2004--2024), filtered to the secid universe from the crosswalk.

**Collection:** The table is in long format `(secid, date, days, borrowrate)`. Filtered to `days IN (10, 30, 60)`. Sentinel values of -99.99 are replaced with NaN. Data is pivoted from long to wide: one row per `(secid, date)` with columns `rate10`, `rate30`, `rate60`.

**Output:**
- Per-year: `Data/Data_Collection/Initial/10_OptionSuite/om_borrow_rates_yearly/om_borrow_rates_YYYY.parquet`
- Combined: `Data/Data_Collection/Initial/10_OptionSuite/om_borrow_rates_all.parquet`

---

## Stage 6: Final Merge & Validation

Merges all option factor files into a single daily stock-level panel keyed on `(permno, date)`.

### Identifier Conversion
Files in secid-space (vol surface, Greeks, borrow rates) are converted to permno-space using the date-aware crosswalk: a left merge on `secid` followed by filtering to rows where `date` falls between `link_start_date` and `link_end_date`.

Files already in permno-space (OptionSuite base, VRP) are merged directly.

### Merge Order
1. **Base:** OptionSuite (`options_filtered.parquet`) -- `iv_catm`, `hvol`, `skew_otm`, `parity_vspread`, `pc_ratio`, moneyness-bucketed OI, etc. Columns `mdate`, `wdate`, `secid` are dropped.
2. **+ VRP:** `om_vrp.parquet` -- only `rv_30d`, `vrp_rv`, `vrp_hvol` (iv_catm and hvol already in base). Left join on `(permno, date)`.
3. **+ Vol Surface:** `om_vol_surface_all.parquet` (converted to permno) -- `iv_91d_atm`, `iv_30d_call25`, `iv_30d_put25`, `vol_term_structure`, `vol_smile`. The `iv_30d_atm_vsurfd` column is dropped as redundant with `iv_catm` from OptionSuite. Left join on `(permno, date)`.
4. **+ Greeks:** `om_greeks_positioning_all.parquet` (converted to permno) -- `gex`, `dex`, `delta_dollar_volume`, OI-weighted Greeks, `total_oi`, `total_volume`. Left join on `(permno, date)`.
5. **+ Borrow Rates:** `om_borrow_rates_all.parquet` (converted to permno) -- `rate10`, `rate30`, `rate60`. Left join on `(permno, date)`.

### Deduplication
Any duplicate `(permno, date)` rows are resolved by keeping the row with the highest `total_oi`.

### Scale Normalisation
Dollar-scale factors (`gex`, `dex`, `delta_dollar_volume`) and contract count factors (`total_oi`, `total_volume`) are divided by market cap (`dlycap` from CRSP daily) to produce `_norm` variants. Moneyness-bucketed OI columns (`sumOI_c_money1/2/3`, `sumOI_p_money1/2/3`) are divided by `total_oi` to produce `_pct` variants. The raw (unnormalised) versions and the market cap helper column are dropped. Columns `nopt_CATM`, `nopt_PATM`, `nopt_POTM` are also dropped.

### Look-Ahead Bias Audit
All factors are verified as valid predictors as of the close of date t for predicting returns from close of t to close of t+1:
- OptionSuite and vsurfd snapshots are recorded at or before market close on date t
- Greeks and positioning are computed from end-of-day open interest and volume for date t
- Borrow rates are date-t implied rates
- `rv_30d` uses returns from t-1 backward (30 trading days)
- VRP factors are the difference of date-t IV and backward-looking realised vol

### Output
- `Data/Data_Collection/Initial/10_OptionSuite/final/om_options_factors_panel.parquet` -- merged panel keyed on `(permno, date)`
- `Data/Data_Collection/Initial/10_OptionSuite/final/om_merge_diagnostics.csv` -- per-column summary statistics (non-null count, null %, mean, std, min, percentiles, max)

In [2]:
import duckdb
import pandas as pd
from pathlib import Path

OUTPUT_DIR = Path('../../Data/Data_Collection/Initial/10_OptionSuite')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

master = pd.read_parquet('../../Data/Data_Collection/Initial/05_Top100_SP500_Universe/universe_master.parquet')
master = master[['permno']].drop_duplicates()

con = duckdb.connect()
con.register('master', master)

con.execute(f"""
    COPY (
        SELECT f.*
        FROM '../../Data/Data_Collection/Initial/10_OptionSuite/optionsuite_download.csv' f
        INNER JOIN master m ON f.PERMNO = m.permno
    )
    TO '{OUTPUT_DIR / "options_filtered.parquet"}' (FORMAT PARQUET)
""")

con.close()

# Quick verification
df = pd.read_parquet(OUTPUT_DIR / 'options_filtered.parquet')
print(f"Shape: {df.shape}")
print(f"Unique PERMNOs: {df['PERMNO'].nunique()}")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")
del df

Shape: (999844, 22)
Unique PERMNOs: 216
Date range: 2003-01-02 to 2025-08-29


# Crosswalk

In [12]:
# %% [markdown]
# # Stage 2: OptionMetrics → CRSP Identifier Crosswalk
#
# Builds a date-aware mapping from OptionMetrics `secid` to CRSP `permno`
# by joining through 8-digit CUSIP, filtered to our top-100 universe only.
#
# Join logic:
#   optionm.secnmd (secid → cusip, valid from effect_date onwards)
#   crsp.stocknames (ncusip → permno, valid namedt to nameendt)
#   Overlap: GREATEST(effect_date, namedt) <= nameendt
#
# Note: secnmd may not have an expiration_date column — we use CRSP's
# nameendt as the end-date boundary instead, which is always present.
#
# Output: ../../Data/Data_Collection/Initial/10_OptionSuite/om_crosswalk.parquet

# %%
import wrds
import pandas as pd
from pathlib import Path
from datetime import datetime

OUTPUT_DIR = Path('../../Data/Data_Collection/Initial/10_OptionSuite')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

conn = wrds.Connection(wrds_username='henrylavender')

# Load master PERMNO list
master = pd.read_parquet('../../Data/Data_Collection/Initial/05_Top100_SP500_Universe/universe_master.parquet')
permno_list = master['permno'].drop_duplicates().tolist()
permnos_sql = tuple(permno_list)
print(f"Master list: {len(permno_list)} unique PERMNOs")

# %% [markdown]
# ## Build the crosswalk via date-aware CUSIP join

# %%
print("Querying OptionMetrics secnmd → CRSP stocknames crosswalk...")

crosswalk = conn.raw_sql(f"""
    SELECT
        a.secid,
        a.cusip,
        b.permno,
        GREATEST(a.effect_date, b.namedt) AS link_start_date,
        b.nameenddt AS link_end_date
    FROM optionm.secnmd a
    INNER JOIN crsp.stocknames b
        ON a.cusip = b.ncusip
    WHERE GREATEST(a.effect_date, b.namedt) <= b.nameenddt
      AND b.permno IN {permnos_sql}
      AND b.nameenddt >= '2004-01-01'
""", date_cols=['link_start_date', 'link_end_date'])

print(f"Crosswalk shape: {crosswalk.shape}")
print(f"\nDescribe:")
print(crosswalk.describe(include='all'))

# %% [markdown]
# ## Check match rate against our universe

# %%
matched_permnos = set(crosswalk['permno'].unique())
master_permnos = set(permno_list)
unmatched = master_permnos - matched_permnos

print(f"\nMaster PERMNOs:     {len(master_permnos):,}")
print(f"Matched via secnmd: {len(matched_permnos):,} ({len(matched_permnos)/len(master_permnos)*100:.1f}%)")
print(f"Unmatched:          {len(unmatched):,}")

if unmatched:
    print(f"Unmatched PERMNOs: {sorted(unmatched)}")

print(f"\nUnique secids mapped: {crosswalk['secid'].nunique():,}")

# %% [markdown]
# ## Save crosswalk

# %%
out_path = OUTPUT_DIR / 'om_crosswalk.parquet'

if out_path.exists():
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    out_path = OUTPUT_DIR / f'om_crosswalk_{timestamp}.parquet'
    print(f"WARNING: om_crosswalk.parquet already exists. Saving as {out_path.name} instead.")

crosswalk.to_parquet(out_path, index=False, engine='pyarrow')
print(f"Saved {out_path.name}: {crosswalk.shape}")

# %%
conn.close()
print("\nStage 2 complete.")

Loading library list...
Done
Master list: 227 unique PERMNOs
Querying OptionMetrics secnmd → CRSP stocknames crosswalk...
Crosswalk shape: (2550, 5)

Describe:
                secid     cusip        permno                link_start_date  \
count          2550.0      2550        2550.0                           2550   
unique           <NA>       302          <NA>                            NaN   
top              <NA>  61744644          <NA>                            NaN   
freq             <NA>        77          <NA>                            NaN   
mean    111158.821176       NaN  49364.953725  2013-06-20 23:19:54.352941056   
min          100892.0       NaN       10104.0            1996-01-01 00:00:00   
25%          103091.0       NaN       19502.0            2006-07-07 00:00:00   
50%          106566.0       NaN       49680.0            2015-06-30 12:00:00   
75%          110752.0       NaN       76841.0            2020-08-03 00:00:00   
max          213638.0       NaN       93

# OptionMetrics

# Volatility Surface

In [16]:
# %% [markdown]
# # Stage 3: Volatility Surface — Term Structure & Smile (vsurfd)
#
# Extracts additional volatility surface factors from OptionMetrics that
# OptionSuite does not provide:
#   - 91-day ATM IV (for volatility term structure)
#   - 25-delta OTM call IV (for volatility smile/convexity)
#
# OptionSuite already provides: ATM call IV (~30d), ATM put IV, OTM put IV, skew.
# We do NOT recompute those here.
#
# Uses conditional aggregation in SQL to produce one row per (secid, date)
# with all raw IVs plus two derived factors:
#   vol_term_structure = iv_91d_atm - iv_30d_atm
#   vol_smile = iv_30d_put25 + iv_30d_call25 - 2 * iv_30d_atm
#
# NOTE: OptionMetrics stores the volatility surface in yearly tables:
#   optionm.vsurfd2004, optionm.vsurfd2005, ..., optionm.vsurfd2024
#
# Output:
#   .../10_OptionSuite/om_vol_surface_yearly/om_vol_surface_YYYY.parquet (per year)
#   .../10_OptionSuite/om_vol_surface_all.parquet (combined)

# %% [markdown]
# ## Setup

# %%
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime

OUTPUT_DIR = Path('../../Data/Data_Collection/Initial/10_OptionSuite')
YEARLY_DIR = OUTPUT_DIR / 'om_vol_surface_yearly'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
YEARLY_DIR.mkdir(parents=True, exist_ok=True)

START_YEAR = 2004
END_YEAR = 2024

# Extract secid universe from the crosswalk built in Stage 2
secid_list = crosswalk['secid'].unique().tolist()
secids_sql = tuple(secid_list)
print(f"Universe: {len(secid_list)} unique secids from crosswalk")

# %% [markdown]
# ## Schema verification
#
# Check against one year's table to confirm column names.

# %%
print("Verifying schema using optionm.vsurfd2020...")
conn = wrds.Connection(wrds_username='henrylavender')

schema = conn.raw_sql("""
    SELECT column_name, data_type
    FROM information_schema.columns
    WHERE table_schema = 'optionm' AND table_name = 'vsurfd2020'
    ORDER BY ordinal_position
""")
print(f"\noptionm.vsurfd2020 — {len(schema)} columns:")
for _, row in schema.iterrows():
    print(f"  {row['column_name']:<30s} {row['data_type']}")

# Quick sample to confirm values
sample = conn.raw_sql(f"""
    SELECT secid, date, days, delta, cp_flag, impl_volatility
    FROM optionm.vsurfd2020
    WHERE secid IN {secids_sql}
      AND date = '2020-01-02'
    LIMIT 20
""")
print(f"\nSample rows:")
print(sample.to_string(index=False))
print(f"\nUnique days values in sample: {sorted(sample['days'].unique())}")
print(f"Unique delta values in sample: {sorted(sample['delta'].unique())}")
print(f"Unique cp_flag values in sample: {sorted(sample['cp_flag'].unique())}")

# %% [markdown]
# ## Check which yearly tables exist

# %%
available_tables = conn.list_tables(library='optionm')
vsurfd_tables = sorted([t for t in available_tables if t.startswith('vsurfd')])
print(f"Available vsurfd tables: {len(vsurfd_tables)}")
print(f"Range: {vsurfd_tables[0]} to {vsurfd_tables[-1]}")

# %% [markdown]
# ## Download year by year with conditional aggregation

# %%
for year in range(START_YEAR, END_YEAR + 1):
    table_name = f'vsurfd{year}'
    out_path = YEARLY_DIR / f'om_vol_surface_{year}.parquet'

    if out_path.exists():
        print(f"  {year}: already exists — skipping")
        continue

    if table_name not in vsurfd_tables:
        print(f"  {year}: table optionm.{table_name} not found — skipping")
        continue

    query = f"""
        SELECT
            secid,
            date,
            MAX(CASE WHEN days = 30 AND delta = 50 AND cp_flag = 'C'
                     THEN impl_volatility END) AS iv_30d_atm_vsurfd,
            MAX(CASE WHEN days = 91 AND delta = 50 AND cp_flag = 'C'
                     THEN impl_volatility END) AS iv_91d_atm,
            MAX(CASE WHEN days = 30 AND delta = 25 AND cp_flag = 'C'
                     THEN impl_volatility END) AS iv_30d_call25,
            MAX(CASE WHEN days = 30 AND delta = -25 AND cp_flag = 'P'
                     THEN impl_volatility END) AS iv_30d_put25
        FROM optionm.{table_name}
        WHERE secid IN {secids_sql}
          AND days IN (30, 91)
          AND delta IN (25, 50, -25)
        GROUP BY secid, date
    """

    df_year = conn.raw_sql(query, date_cols=['date'])

    if df_year.empty:
        print(f"  {year}: 0 rows — skipping")
        continue

    # Compute derived factors
    df_year['vol_term_structure'] = df_year['iv_91d_atm'] - df_year['iv_30d_atm_vsurfd']
    df_year['vol_smile'] = (
        df_year['iv_30d_put25'] + df_year['iv_30d_call25']
        - 2 * df_year['iv_30d_atm_vsurfd']
    )

    # Sanity checks
    iv_cols = ['iv_30d_atm_vsurfd', 'iv_91d_atm', 'iv_30d_call25', 'iv_30d_put25']
    null_pct = (df_year[iv_cols].isna().mean() * 100).round(1)

    for col in iv_cols:
        non_null = df_year[col].dropna()
        n_bad = (non_null <= 0).sum()
        if n_bad > 0:
            print(f"  WARNING: {col} has {n_bad} non-positive values in {year}")

    print(f"  {year}: {len(df_year):,} rows, {df_year['secid'].nunique()} secids | "
          f"NaN%: {null_pct.to_dict()}")

    df_year.to_parquet(out_path, index=False, engine='pyarrow')

print("\nPer-year downloads complete.")

# %% [markdown]
# ## Concatenate all years

# %%
year_files = sorted(YEARLY_DIR.glob('om_vol_surface_2*.parquet'))
print(f"Loading {len(year_files)} year files...")

df = pd.concat([pd.read_parquet(f) for f in year_files], ignore_index=True)
df = df.sort_values(['secid', 'date']).reset_index(drop=True)

# %% [markdown]
# ## Full-sample sanity checks

# %%
print(f"Total shape: {df.shape}")
print(f"Date range: {df['date'].min().date()} to {df['date'].max().date()}")
print(f"Unique secids: {df['secid'].nunique():,}")

print(f"\nvol_term_structure (should be ~0 mean, negative in stress):")
print(f"  Mean:   {df['vol_term_structure'].mean():.4f}")
print(f"  Median: {df['vol_term_structure'].median():.4f}")

print(f"\nvol_smile (should be positive on average):")
print(f"  Mean:   {df['vol_smile'].mean():.4f}")
print(f"  Median: {df['vol_smile'].median():.4f}")

print(f"\nNull % per column:")
null_pct = (df.isna().mean() * 100).round(1)
for col, pct in null_pct.items():
    print(f"  {col:<25s} {pct:5.1f}%")

# %% [markdown]
# ## Save combined file

# %%
out_path = OUTPUT_DIR / 'om_vol_surface_all.parquet'

if out_path.exists():
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    out_path = OUTPUT_DIR / f'om_vol_surface_all_{timestamp}.parquet'
    print(f"WARNING: om_vol_surface_all.parquet already exists. Saving as {out_path.name}")

df.to_parquet(out_path, index=False, engine='pyarrow')
print(f"Saved {out_path.name}: {df.shape}")

del df
print("\nStage 3 complete.")

Universe: 229 unique secids from crosswalk
Verifying schema using optionm.vsurfd2020...
Loading library list...
Done

optionm.vsurfd2020 — 9 columns:
  secid                          double precision
  date                           date
  days                           double precision
  delta                          double precision
  impl_volatility                double precision
  impl_strike                    double precision
  impl_premium                   double precision
  dispersion                     double precision
  cp_flag                        character varying

Sample rows:
    secid       date  days  delta cp_flag  impl_volatility
 100892.0 2020-01-02  10.0  -90.0       P         0.388435
 100892.0 2020-01-02  10.0  -85.0       P          0.39597
 100892.0 2020-01-02  10.0  -80.0       P         0.380534
 100892.0 2020-01-02  10.0  -75.0       P         0.353697
 100892.0 2020-01-02  10.0  -70.0       P         0.334923
 100892.0 2020-01-02  10.0  -65.0       P  

In [25]:
# %% [markdown]
# # Stage 4: Greeks & Positioning Factors (opprcd)
#
# Computes daily stock-level Greeks-based positioning factors from OptionMetrics
# option price tables. This is the most data-intensive stage — all heavy
# computation happens on the WRDS Postgres server via SQL.
#
# Factors computed (aggregated to secid × date):
#   gex                — Gamma Exposure (calls - puts), measures dealer hedging pressure
#   dex                — Delta Exposure, net directional positioning
#   delta_dollar_volume — Flow toxicity, delta-weighted dollar volume
#   oi_wt_delta        — OI-weighted average delta (net positioning tilt)
#   oi_wt_gamma        — OI-weighted average gamma (convexity exposure)
#   oi_wt_vega         — OI-weighted average vega (vol sensitivity)
#   oi_wt_theta        — OI-weighted average theta (time decay exposure)
#   total_oi           — Total open interest (diagnostic)
#   total_volume       — Total volume (diagnostic)
#
# Source: optionm.opprcd{YYYY} joined with optionm.secprd (for stock price)
#
# Output:
#   .../10_OptionSuite/om_greeks_positioning_yearly/om_greeks_positioning_YYYY.parquet
#   .../10_OptionSuite/om_greeks_positioning_all.parquet

# %% [markdown]
# ## Setup

# %%
import wrds
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime

OUTPUT_DIR = Path('../../Data/Data_Collection/Initial/10_OptionSuite')
YEARLY_DIR = OUTPUT_DIR / 'om_greeks_positioning_yearly'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
YEARLY_DIR.mkdir(parents=True, exist_ok=True)

START_YEAR = 2004
END_YEAR = 2024

conn = wrds.Connection(wrds_username='henrylavender')

# Extract secid universe from the crosswalk built in Stage 2
secid_list = crosswalk['secid'].unique().tolist()
secids_sql = tuple(secid_list)
print(f"Universe: {len(secid_list)} unique secids from crosswalk")

# %% [markdown]
# ## Check which opprcd yearly tables exist

# %%
available_tables = conn.list_tables(library='optionm')
opprcd_tables = sorted([t for t in available_tables if t.startswith('opprcd')])
print(f"Available opprcd tables: {len(opprcd_tables)}")
print(f"Range: {opprcd_tables[0]} to {opprcd_tables[-1]}")

# %% [markdown]
# ## Download year by year
#
# Uses a CTE to compute per-contract intermediates, then aggregates
# to (secid, date) in the outer query. All computation runs on WRDS.

# %%
for year in range(START_YEAR, END_YEAR + 1):
    table_name = f'opprcd{year}'
    out_path = YEARLY_DIR / f'om_greeks_positioning_{year}.parquet'

    if out_path.exists():
        print(f"  {year}: already exists — skipping")
        continue

    if table_name not in opprcd_tables:
        print(f"  {year}: table optionm.{table_name} not found — skipping")
        continue

    query = f"""
        WITH contracts AS (
            SELECT
                o.secid,
                o.date,
                o.cp_flag,
                o.delta,
                o.gamma,
                o.vega,
                o.theta,
                o.open_interest,
                o.volume,
                s.close AS stock_close,

                -- Per-contract GEX: gamma * OI * contract_size * price^2 * 0.01
                o.gamma * o.open_interest * COALESCE(o.contract_size, 100)
                    * s.close * s.close * 0.01
                    AS gex_contract,

                -- Per-contract DEX: delta * OI * contract_size * price
                o.delta * o.open_interest * COALESCE(o.contract_size, 100)
                    * s.close
                    AS dex_contract,

                -- Per-contract flow toxicity: delta * volume * contract_size * price
                CASE WHEN o.volume > 0
                     THEN o.delta * o.volume * COALESCE(o.contract_size, 100) * s.close
                     ELSE 0
                END AS toxicity_contract

            FROM optionm.{table_name} o
            INNER JOIN optionm.secprd s
                ON o.secid = s.secid AND o.date = s.date
            WHERE o.secid IN {secids_sql}
              AND o.open_interest > 0
              AND o.impl_volatility IS NOT NULL
              AND o.delta IS NOT NULL AND o.delta BETWEEN -1 AND 1
              AND o.gamma IS NOT NULL
              AND s.close IS NOT NULL AND s.close > 0
        )
        SELECT
            secid,
            date,

            -- GEX: call gamma minus put gamma
            SUM(CASE WHEN cp_flag = 'C' THEN gex_contract ELSE 0 END)
            - SUM(CASE WHEN cp_flag = 'P' THEN gex_contract ELSE 0 END)
                AS gex,

            -- DEX: net delta exposure
            SUM(dex_contract) AS dex,

            -- Flow toxicity
            SUM(toxicity_contract) AS delta_dollar_volume,

            -- OI-weighted Greeks
            SUM(delta * open_interest) / NULLIF(SUM(open_interest), 0)
                AS oi_wt_delta,
            SUM(gamma * open_interest) / NULLIF(SUM(open_interest), 0)
                AS oi_wt_gamma,
            SUM(vega * open_interest) / NULLIF(SUM(open_interest), 0)
                AS oi_wt_vega,
            SUM(theta * open_interest) / NULLIF(SUM(open_interest), 0)
                AS oi_wt_theta,

            -- Diagnostics
            SUM(open_interest) AS total_oi,
            SUM(volume) AS total_volume

        FROM contracts
        GROUP BY secid, date
    """

    print(f"  {year}: querying...", end=' ', flush=True)
    df_year = conn.raw_sql(query, date_cols=['date'])

    if df_year.empty:
        print("0 rows — skipping")
        continue

    # Sanity checks
    n_rows = len(df_year)
    n_secids = df_year['secid'].nunique()

    # Flag extreme GEX values
    n_extreme = (df_year['gex'].abs() > 1e12).sum()
    extreme_msg = f" | WARNING: {n_extreme} extreme GEX values" if n_extreme > 0 else ""

    # Null counts
    null_counts = df_year.isna().sum()
    cols_with_nulls = null_counts[null_counts > 0]

    print(f"{n_rows:,} rows, {n_secids} secids{extreme_msg}")

    if len(cols_with_nulls) > 0:
        print(f"    Nulls: {cols_with_nulls.to_dict()}")

    df_year.to_parquet(out_path, index=False, engine='pyarrow')

print("\nPer-year downloads complete.")

# %% [markdown]
# ## Concatenate all years

# %%
year_files = sorted(YEARLY_DIR.glob('om_greeks_positioning_2*.parquet'))
print(f"Loading {len(year_files)} year files...")

df = pd.concat([pd.read_parquet(f) for f in year_files], ignore_index=True)
df = df.sort_values(['secid', 'date']).reset_index(drop=True)

# %% [markdown]
# ## Full-sample sanity checks

# %%
print(f"Total shape: {df.shape}")
print(f"Date range: {df['date'].min().date()} to {df['date'].max().date()}")
print(f"Unique secids: {df['secid'].nunique():,}")

print(f"\nDescribe:")
print(df.describe().to_string())

print(f"\nNull % per column:")
null_pct = (df.isna().mean() * 100).round(1)
for col, pct in null_pct.items():
    print(f"  {col:<25s} {pct:5.1f}%")

n_extreme = (df['gex'].abs() > 1e12).sum()
print(f"\nExtreme GEX (|gex| > 1e12): {n_extreme} rows ({n_extreme/len(df)*100:.2f}%)")

print(f"\n5 random rows:")
print(df.sample(5, random_state=42).to_string(index=False))

# %% [markdown]
# ## Save combined file

# %%
out_path = OUTPUT_DIR / 'om_greeks_positioning_all.parquet'

if out_path.exists():
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    out_path = OUTPUT_DIR / f'om_greeks_positioning_all_{timestamp}.parquet'
    print(f"WARNING: om_greeks_positioning_all.parquet already exists. Saving as {out_path.name}")

df.to_parquet(out_path, index=False, engine='pyarrow')
print(f"Saved {out_path.name}: {df.shape}")

del df
print("\nStage 4 complete.")

Loading library list...
Done
Universe: 229 unique secids from crosswalk
Available opprcd tables: 30
Range: opprcd1996 to opprcd2025
  2004: querying... 49,507 rows, 199 secids
  2005: querying... 49,476 rows, 197 secids
  2006: querying... 48,796 rows, 196 secids
  2007: querying... 49,104 rows, 198 secids
  2008: querying... 49,763 rows, 199 secids
  2009: querying... 48,547 rows, 195 secids
  2010: querying... 47,976 rows, 195 secids
  2011: querying... 48,446 rows, 193 secids
  2012: querying... 48,557 rows, 197 secids
  2013: querying... 49,689 rows, 198 secids
  2014: querying... 49,439 rows, 197 secids
  2015: querying... 49,061 rows, 198 secids
  2016: querying... 48,639 rows, 195 secids
  2017: querying... 47,992 rows, 193 secids
  2018: querying... 47,384 rows, 191 secids
  2019: querying... 46,934 rows, 189 secids
  2020: querying... 46,189 rows, 184 secids
  2021: querying... 45,858 rows, 182 secids
  2022: querying... 45,676 rows, 182 secids
  2023: querying... 45,495 rows,

In [31]:
# %% [markdown]
# # Stage 5: Volatility Risk Premium (VRP) & Implied Borrow Rates
#
# Part A: Computes VRP as the spread between implied and realised volatility.
#   vrp_hvol = iv_catm - hvol  (OptionSuite's matched 30d historical vol)
#   vrp_rv   = iv_catm - rv_30d (independently computed from CRSP returns)
#
# Part B: Downloads implied borrow rates from optionm.stdbrte{YYYY}.
#   Table is in long format: (secid, date, days, borrowrate).
#   We filter to 10/30/60-day tenors, replace -99.99 sentinels with NaN,
#   and pivot to wide format: one row per (secid, date) with rate10/rate30/rate60.
#
# Output:
#   .../10_OptionSuite/om_vrp.parquet
#   .../10_OptionSuite/om_borrow_rates_yearly/om_borrow_rates_YYYY.parquet
#   .../10_OptionSuite/om_borrow_rates_all.parquet

# %% [markdown]
# ## Setup

# %%
import wrds
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime

OUTPUT_DIR = Path('../../Data/Data_Collection/Initial/10_OptionSuite')
BORROW_YEARLY_DIR = OUTPUT_DIR / 'om_borrow_rates_yearly'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
BORROW_YEARLY_DIR.mkdir(parents=True, exist_ok=True)

START_YEAR = 2004
END_YEAR = 2024

# %% [markdown]
# ## Part A: Volatility Risk Premium

# %% [markdown]
# ### A1. Load OptionSuite data

# %%
print("Loading OptionSuite data...")
opts = pd.read_parquet(OUTPUT_DIR / 'options_filtered.parquet')
opts['date'] = pd.to_datetime(opts['date'])

opts = opts.rename(columns={'PERMNO': 'permno', 'iv_CATM': 'iv_catm'})

print(f"OptionSuite shape: {opts.shape}")
print(f"Columns: {opts.columns.tolist()}")

# %% [markdown]
# ### A2. Unit check — are IVs in decimal or percentage?

# %%
print("\nUnit check:")
print(f"  iv_catm mean: {opts['iv_catm'].mean():.4f}")
print(f"  hvol mean:    {opts['hvol'].mean():.4f}")

if opts['iv_catm'].mean() > 1.0:
    print("  → iv_catm appears to be in percentage terms, dividing by 100")
    opts['iv_catm'] = opts['iv_catm'] / 100

if opts['hvol'].mean() > 1.0:
    print("  → hvol appears to be in percentage terms, dividing by 100")
    opts['hvol'] = opts['hvol'] / 100

print(f"\nAfter conversion:")
print(f"  iv_catm mean: {opts['iv_catm'].mean():.4f}")
print(f"  hvol mean:    {opts['hvol'].mean():.4f}")

# %% [markdown]
# ### A3. Compute rv_30d from CRSP daily returns
#
# Trailing 30-trading-day realised volatility (annualised, backward-looking).
# Uses the prior 30 trading days NOT including today's return to avoid lookahead.

# %%
print("\nLoading CRSP daily data...")
crsp = pd.read_parquet('../../Data/Data_Collection/Initial/06_Daily_CRSP_Stock_Data/firm_daily',
                        columns=['permno', 'date', 'dlyret'])
crsp['date'] = pd.to_datetime(crsp['date'])
crsp = crsp.sort_values(['permno', 'date']).reset_index(drop=True)

print(f"CRSP shape: {crsp.shape}")

print("Computing rv_30d (trailing 30-day realised vol, annualised)...")
crsp['ret_shifted'] = crsp.groupby('permno')['dlyret'].shift(1)
crsp['rv_30d'] = (
    crsp.groupby('permno')['ret_shifted']
    .rolling(30, min_periods=20)
    .std()
    .reset_index(level=0, drop=True)
    * np.sqrt(252)
)
crsp = crsp.drop(columns=['ret_shifted', 'dlyret'])
crsp = crsp.dropna(subset=['rv_30d'])

print(f"rv_30d computed: {len(crsp):,} rows")
print(f"rv_30d mean: {crsp['rv_30d'].mean():.4f}")

if crsp['rv_30d'].mean() > 1.0:
    print("  → rv_30d appears to be in percentage terms, dividing by 100")
    crsp['rv_30d'] = crsp['rv_30d'] / 100

# %% [markdown]
# ### A4. Compute VRP

# %%
opts['vrp_hvol'] = opts['iv_catm'] - opts['hvol']

print("Merging OptionSuite with CRSP rv_30d...")
vrp = opts[['permno', 'date', 'iv_catm', 'hvol', 'vrp_hvol']].copy()
vrp = vrp.merge(crsp[['permno', 'date', 'rv_30d']], on=['permno', 'date'], how='inner')
vrp['vrp_rv'] = vrp['iv_catm'] - vrp['rv_30d']

print(f"VRP panel shape: {vrp.shape}")
print(f"Unique permnos: {vrp['permno'].nunique()}")
print(f"Date range: {vrp['date'].min().date()} to {vrp['date'].max().date()}")

# %% [markdown]
# ### A5. Sanity checks

# %%
print("\n--- VRP Sanity Checks ---")

print(f"\nvrp_hvol (should be positive on average):")
print(f"  Mean:   {vrp['vrp_hvol'].mean():.4f}")
print(f"  Median: {vrp['vrp_hvol'].median():.4f}")
print(f"  % negative: {(vrp['vrp_hvol'] < 0).mean() * 100:.1f}%")

print(f"\nvrp_rv (should be positive on average):")
print(f"  Mean:   {vrp['vrp_rv'].mean():.4f}")
print(f"  Median: {vrp['vrp_rv'].median():.4f}")
print(f"  % negative: {(vrp['vrp_rv'] < 0).mean() * 100:.1f}%")

corr = vrp['vrp_hvol'].corr(vrp['vrp_rv'])
print(f"\nCorrelation between vrp_hvol and vrp_rv: {corr:.4f}")
if corr < 0.5:
    print("  WARNING: Low correlation — check units or window definitions")

n_extreme_hvol = (vrp['vrp_hvol'].abs() > 1.0).sum()
n_extreme_rv = (vrp['vrp_rv'].abs() > 1.0).sum()
print(f"\n|vrp_hvol| > 1.0: {n_extreme_hvol} rows ({n_extreme_hvol/len(vrp)*100:.2f}%)")
print(f"|vrp_rv| > 1.0:   {n_extreme_rv} rows ({n_extreme_rv/len(vrp)*100:.2f}%)")

# %% [markdown]
# ### A6. Save VRP

# %%
vrp_out = vrp[['permno', 'date', 'iv_catm', 'rv_30d', 'hvol', 'vrp_rv', 'vrp_hvol']]

out_path = OUTPUT_DIR / 'om_vrp.parquet'
if out_path.exists():
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    out_path = OUTPUT_DIR / f'om_vrp_{timestamp}.parquet'
    print(f"WARNING: om_vrp.parquet already exists. Saving as {out_path.name}")

vrp_out.to_parquet(out_path, index=False, engine='pyarrow')
print(f"Saved {out_path.name}: {vrp_out.shape}")

del opts, crsp, vrp, vrp_out

# %% [markdown]
# ## Part B: Implied Borrow Rates

# %%
conn = wrds.Connection(wrds_username='henrylavender')

secid_list = crosswalk['secid'].unique().tolist()
secids_sql = tuple(secid_list)
print(f"Universe: {len(secid_list)} unique secids")

# %% [markdown]
# ### B1. Check which stdbrte yearly tables exist

# %%
available_tables = conn.list_tables(library='optionm')
stdbrte_tables = sorted([t for t in available_tables if t.startswith('stdbrte')])
print(f"Available stdbrte tables: {len(stdbrte_tables)}")
print(f"Range: {stdbrte_tables[0]} to {stdbrte_tables[-1]}")

# %% [markdown]
# ### B2. Download borrow rates year by year

# %%
for year in range(START_YEAR, END_YEAR + 1):
    table_name = f'stdbrte{year}'
    out_path = BORROW_YEARLY_DIR / f'om_borrow_rates_{year}.parquet'

    if out_path.exists():
        print(f"  {year}: already exists — skipping")
        continue

    if table_name not in stdbrte_tables:
        print(f"  {year}: table optionm.{table_name} not found — skipping")
        continue

    df_year = conn.raw_sql(f"""
        SELECT secid, date, days, borrowrate
        FROM optionm.{table_name}
        WHERE secid IN {secids_sql}
          AND days IN (10, 30, 60)
    """, date_cols=['date'])

    if df_year.empty:
        print(f"  {year}: 0 rows — skipping")
        continue

    # Replace -99.99 sentinel values with NaN
    df_year.loc[df_year['borrowrate'] <= -99, 'borrowrate'] = np.nan

    # Pivot from long to wide: (secid, date) with rate10, rate30, rate60
    df_year['days_col'] = 'rate' + df_year['days'].astype(int).astype(str)
    df_year = df_year.pivot_table(
        index=['secid', 'date'],
        columns='days_col',
        values='borrowrate',
        aggfunc='first'
    ).reset_index()
    df_year.columns.name = None

    # Deduplicate
    n_dupes = df_year.duplicated(subset=['secid', 'date']).sum()
    if n_dupes > 0:
        df_year = df_year.drop_duplicates(subset=['secid', 'date'], keep='first')

    rate_cols = [c for c in ['rate10', 'rate30', 'rate60'] if c in df_year.columns]
    null_pct = {col: f"{df_year[col].isna().mean()*100:.1f}%" for col in rate_cols}
    n_htb = (df_year['rate10'] > 10).sum() if 'rate10' in df_year.columns else 0
    dupe_msg = f", {n_dupes} dupes removed" if n_dupes > 0 else ""

    print(f"  {year}: {len(df_year):,} rows, {df_year['secid'].nunique()} secids | "
          f"NaN%: {null_pct} | HTB(rate10>10%): {n_htb}{dupe_msg}")

    df_year.to_parquet(out_path, index=False, engine='pyarrow')

print("\nPer-year downloads complete.")

# %% [markdown]
# ### B3. Concatenate all years

# %%
year_files = sorted(BORROW_YEARLY_DIR.glob('om_borrow_rates_2*.parquet'))
print(f"Loading {len(year_files)} year files...")

borrow = pd.concat([pd.read_parquet(f) for f in year_files], ignore_index=True)
borrow = borrow.sort_values(['secid', 'date']).reset_index(drop=True)

# %% [markdown]
# ### B4. Full-sample sanity checks

# %%
print(f"Total shape: {borrow.shape}")
print(f"Date range: {borrow['date'].min().date()} to {borrow['date'].max().date()}")
print(f"Unique secids: {borrow['secid'].nunique():,}")

print(f"\nDescribe:")
print(borrow.describe().to_string())

rate_cols = [c for c in ['rate10', 'rate30', 'rate60'] if c in borrow.columns]

print(f"\nNull % per column:")
for col in rate_cols:
    pct = borrow[col].isna().mean() * 100
    print(f"  {col:<10s} {pct:5.1f}%")

for col in rate_cols:
    n_htb = (borrow[col] > 10).sum()
    print(f"  {col} > 10% (hard to borrow): {n_htb:,} stock-days")

# %% [markdown]
# ### B5. Save combined borrow rates

# %%
out_path = OUTPUT_DIR / 'om_borrow_rates_all.parquet'
if out_path.exists():
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    out_path = OUTPUT_DIR / f'om_borrow_rates_all_{timestamp}.parquet'
    print(f"WARNING: om_borrow_rates_all.parquet already exists. Saving as {out_path.name}")

borrow.to_parquet(out_path, index=False, engine='pyarrow')
print(f"Saved {out_path.name}: {borrow.shape}")

conn.close()
del borrow
print("\nStage 5 complete.")

Loading OptionSuite data...
OptionSuite shape: (999844, 22)
Columns: ['permno', 'secid', 'date', 'iv_catm', 'nopt_CATM', 'iv_PATM', 'nopt_PATM', 'iv_POTM', 'nopt_POTM', 'Skew_OTM', 'Parity_VSpread', 'nopt_Parity', 'PC_Ratio', 'sumOI_c_money1', 'sumOI_c_money2', 'sumOI_c_money3', 'sumOI_p_money1', 'sumOI_p_money2', 'sumOI_p_money3', 'hvol', 'mdate', 'wdate']

Unit check:
  iv_catm mean: 0.2834
  hvol mean:    0.2735

After conversion:
  iv_catm mean: 0.2834
  hvol mean:    0.2735

Loading CRSP daily data...
CRSP shape: (1010280, 3)
Computing rv_30d (trailing 30-day realised vol, annualised)...
rv_30d computed: 1,003,941 rows
rv_30d mean: 0.2845
Merging OptionSuite with CRSP rv_30d...
VRP panel shape: (918215, 7)
Unique permnos: 216
Date range: 2004-02-02 to 2024-12-31

--- VRP Sanity Checks ---

vrp_hvol (should be positive on average):
  Mean:   0.0098
  Median: 0.0214
  % negative: 36.2%

vrp_rv (should be positive on average):
  Mean:   0.0065
  Median: 0.0163
  % negative: 38.7%

Co

In [1]:
# %% [markdown]
# # Stage 6: Final Merge & Validation
#
# Merges all option factor files into a single daily stock-level panel
# keyed on (permno, date).
#
# Files in permno-space (direct merge):
#   options_filtered.parquet  — OptionSuite base (iv_catm, hvol, skew, OI, etc.)
#   om_vrp.parquet            — VRP factors (vrp_rv, vrp_hvol, rv_30d)
#
# Files in secid-space (need crosswalk → permno):
#   om_vol_surface_all.parquet         — vol term structure, smile
#   om_greeks_positioning_all.parquet  — GEX, DEX, OI-weighted Greeks
#   om_borrow_rates_all.parquet        — implied borrow rates
#
# Output:
#   .../10_OptionSuite/final/om_options_factors_panel.parquet
#   .../10_OptionSuite/final/om_merge_diagnostics.csv

# %% [markdown]
# ## Setup

# %%
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime

OUTPUT_DIR = Path('../../Data/Data_Collection/Initial/10_OptionSuite')
FINAL_DIR = OUTPUT_DIR / 'final'
FINAL_DIR.mkdir(parents=True, exist_ok=True)

# %% [markdown]
# ## Step 1: Load crosswalk

# %%
xwalk = pd.read_parquet(OUTPUT_DIR / 'om_crosswalk.parquet')
xwalk['link_start_date'] = pd.to_datetime(xwalk['link_start_date'])
xwalk['link_end_date'] = pd.to_datetime(xwalk['link_end_date'])
xwalk['permno'] = xwalk['permno'].astype(int)
print(f"Crosswalk: {len(xwalk):,} rows, {xwalk['secid'].nunique()} secids, "
      f"{xwalk['permno'].nunique()} permnos")

# %% [markdown]
# ## Step 2: Helper function to convert secid-space → permno-space

# %%
def secid_to_permno(df, xwalk, name=''):
    """Convert a secid-space DataFrame to permno-space using date-aware crosswalk.
    
    Two-step pattern to avoid memory explosion:
    1. Left merge on secid only (small fan-out)
    2. Immediately filter to valid date range, then drop crosswalk columns
    """
    n_before = len(df)
    
    # Step 1: merge on secid
    merged = df.merge(
        xwalk[['secid', 'permno', 'link_start_date', 'link_end_date']],
        on='secid',
        how='left'
    )
    
    # Step 2: filter to valid date range
    merged = merged[
        merged['date'].between(merged['link_start_date'], merged['link_end_date'])
    ]
    
    # Report match rate
    n_matched = len(merged)
    n_unmatched = n_before - df['secid'].isin(merged['secid'].unique()).sum()
    match_pct = merged['secid'].nunique() / df['secid'].nunique() * 100
    print(f"  {name}: {n_before:,} rows → {n_matched:,} matched "
          f"({match_pct:.1f}% of secids mapped)")
    
    # Drop crosswalk columns, keep permno
    merged = merged.drop(columns=['secid', 'cusip', 'link_start_date', 'link_end_date'],
                         errors='ignore')
    merged['permno'] = merged['permno'].astype(int)
    
    return merged

# %% [markdown]
# ## Step 3: Load and convert secid-space files

# %% [markdown]
# ### Vol Surface

# %%
print("Converting vol surface to permno-space...")
vol_surf = pd.read_parquet(OUTPUT_DIR / 'om_vol_surface_all.parquet')
vol_surf['date'] = pd.to_datetime(vol_surf['date'])
vol_surf = secid_to_permno(vol_surf, xwalk, name='vol_surface')

# Drop iv_30d_atm_vsurfd — redundant with iv_catm from OptionSuite
# (may differ slightly: vsurfd uses standardised surface interpolation,
#  OptionSuite uses nearest-ATM contract. Both are valid 30d ATM IV measures.)
vol_surf = vol_surf.drop(columns=['iv_30d_atm_vsurfd'], errors='ignore')
print(f"  Kept columns: {[c for c in vol_surf.columns if c not in ['permno', 'date']]}")

# %% [markdown]
# ### Greeks & Positioning

# %%
print("\nConverting Greeks to permno-space...")
greeks = pd.read_parquet(OUTPUT_DIR / 'om_greeks_positioning_all.parquet')
greeks['date'] = pd.to_datetime(greeks['date'])
greeks = secid_to_permno(greeks, xwalk, name='greeks')
print(f"  Kept columns: {[c for c in greeks.columns if c not in ['permno', 'date']]}")

# %% [markdown]
# ### Borrow Rates

# %%
print("\nConverting borrow rates to permno-space...")
borrow = pd.read_parquet(OUTPUT_DIR / 'om_borrow_rates_all.parquet')
borrow['date'] = pd.to_datetime(borrow['date'])
borrow = secid_to_permno(borrow, xwalk, name='borrow_rates')
print(f"  Kept columns: {[c for c in borrow.columns if c not in ['permno', 'date']]}")

del xwalk

# %% [markdown]
# ## Step 4: Load permno-space files

# %%
print("\nLoading OptionSuite base...")
base = pd.read_parquet(OUTPUT_DIR / 'options_filtered.parquet')
base['date'] = pd.to_datetime(base['date'])
base = base.rename(columns={'PERMNO': 'permno', 'iv_CATM': 'iv_catm'})
base['permno'] = base['permno'].astype(int)

# Drop non-factor columns (mdate, wdate are OptionSuite metadata)
base = base.drop(columns=['mdate', 'wdate', 'secid'], errors='ignore')
print(f"  Base shape: {base.shape}, {base['permno'].nunique()} permnos")

print("\nLoading VRP...")
vrp = pd.read_parquet(OUTPUT_DIR / 'om_vrp.parquet')
vrp['date'] = pd.to_datetime(vrp['date'])
vrp['permno'] = vrp['permno'].astype(int)

# Keep only the new columns — iv_catm and hvol already in base
vrp = vrp[['permno', 'date', 'rv_30d', 'vrp_rv', 'vrp_hvol']]
print(f"  VRP shape: {vrp.shape}")

# %% [markdown]
# ## Step 5: Sequential merge

# %%
print("\n--- Sequential Merge ---")

# Start with OptionSuite base
panel = base.copy()
del base
print(f"Base:          {panel.shape}")

# Merge VRP
panel = panel.merge(vrp, on=['permno', 'date'], how='left')
del vrp
print(f"+ VRP:         {panel.shape}")

# Merge vol surface
panel = panel.merge(vol_surf, on=['permno', 'date'], how='left')
del vol_surf
print(f"+ Vol Surface: {panel.shape}")

# Merge Greeks
panel = panel.merge(greeks, on=['permno', 'date'], how='left')
del greeks
print(f"+ Greeks:      {panel.shape}")

# Merge borrow rates
panel = panel.merge(borrow, on=['permno', 'date'], how='left')
del borrow
print(f"+ Borrow:      {panel.shape}")

# %% [markdown]
# ## Step 6: Duplicate check & resolution

# %%
n_dupes = panel.duplicated(subset=['permno', 'date']).sum()
print(f"\nDuplicate (permno, date) rows: {n_dupes}")

if n_dupes > 0:
    print("Resolving: keeping row with highest total_oi per (permno, date)...")
    # Sort so highest total_oi is first, then deduplicate
    panel = panel.sort_values(['permno', 'date', 'total_oi'], ascending=[True, True, False])
    panel = panel.drop_duplicates(subset=['permno', 'date'], keep='first')
    print(f"After dedup: {panel.shape}")


# %% [markdown]
# ## Step 6b: Normalise scale-dependent factors

# %%
print("Normalising scale-dependent factors...")

crsp_cap = pd.read_parquet(
    '../../Data/Data_Collection/Initial/06_Daily_CRSP_Stock_Data/firm_daily',
    columns=['permno', 'date', 'dlycap']
)
crsp_cap['date'] = pd.to_datetime(crsp_cap['date'])
crsp_cap['permno'] = crsp_cap['permno'].astype(int)

panel = panel.merge(crsp_cap, on=['permno', 'date'], how='left')
del crsp_cap

# Dollar-scale factors → divide by market cap
for col in ['gex', 'dex', 'delta_dollar_volume']:
    panel[f'{col}_norm'] = panel[col] / panel['dlycap'].replace(0, np.nan)

# Contract counts → divide by market cap
for col in ['total_oi', 'total_volume']:
    panel[f'{col}_norm'] = panel[col] / panel['dlycap'].replace(0, np.nan)

# Moneyness OI buckets → % of total OI
for col in ['sumOI_c_money1', 'sumOI_c_money2', 'sumOI_c_money3',
            'sumOI_p_money1', 'sumOI_p_money2', 'sumOI_p_money3']:
    panel[f'{col}_pct'] = panel[col] / panel['total_oi'].replace(0, np.nan)

# Drop raw versions and market cap helper
panel = panel.drop(columns=[
    'gex', 'dex', 'delta_dollar_volume', 'total_oi', 'total_volume',
    'sumOI_c_money1', 'sumOI_c_money2', 'sumOI_c_money3',
    'sumOI_p_money1', 'sumOI_p_money2', 'sumOI_p_money3',
    'nopt_CATM', 'nopt_PATM', 'nopt_POTM',
    'dlycap'
], errors='ignore')

print(f"After normalisation: {panel.shape}")





# %% [markdown]
# ## Step 7: Sort and final shape

# %%
panel = panel.sort_values(['permno', 'date']).reset_index(drop=True)

print(f"\n--- Final Panel ---")
print(f"Shape: {panel.shape}")
print(f"Unique permnos: {panel['permno'].nunique()}")
print(f"Unique dates: {panel['date'].nunique():,}")
print(f"Date range: {panel['date'].min().date()} to {panel['date'].max().date()}")

# %% [markdown]
# ## Step 8: Null analysis

# %%
print(f"\nNull % per column (sorted descending):")
null_pct = (panel.isna().mean() * 100).round(2)
null_pct_sorted = null_pct.sort_values(ascending=False)
for col, pct in null_pct_sorted.items():
    print(f"  {col:<30s} {pct:6.2f}%")

# %% [markdown]
# ## Step 9: Column inventory

# %%
print(f"\nColumn inventory ({len(panel.columns)} columns):")
for col in panel.columns:
    print(f"  {col:<30s} {panel[col].dtype}")

# %% [markdown]
# ## Step 10: Save final panel

# %%
out_path = FINAL_DIR / 'om_options_factors_panel.parquet'
if out_path.exists():
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    out_path = FINAL_DIR / f'om_options_factors_panel_{timestamp}.parquet'
    print(f"WARNING: file already exists. Saving as {out_path.name}")

panel.to_parquet(out_path, index=False, engine='pyarrow')
print(f"Saved {out_path.name}: {panel.shape}")

# %% [markdown]
# ## Step 11: Save diagnostics CSV

# %%
factor_cols = [c for c in panel.columns if c not in ['permno', 'date']]

diag_rows = []
for col in factor_cols:
    s = pd.to_numeric(panel[col], errors='coerce')
    diag_rows.append({
        'column': col,
        'non_null_count': s.notna().sum(),
        'null_pct': round(s.isna().mean() * 100, 2),
        'mean': round(s.mean(), 6) if s.notna().any() else None,
        'std': round(s.std(), 6) if s.notna().any() else None,
        'min': round(s.min(), 6) if s.notna().any() else None,
        'p25': round(s.quantile(0.25), 6) if s.notna().any() else None,
        'median': round(s.quantile(0.5), 6) if s.notna().any() else None,
        'p75': round(s.quantile(0.75), 6) if s.notna().any() else None,
        'max': round(s.max(), 6) if s.notna().any() else None,
    })

diag = pd.DataFrame(diag_rows)

diag_path = FINAL_DIR / 'om_merge_diagnostics.csv'
if diag_path.exists():
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    diag_path = FINAL_DIR / f'om_merge_diagnostics_{timestamp}.csv'

diag.to_csv(diag_path, index=False)
print(f"Saved {diag_path.name}")

# %% [markdown]
# ## Step 12: Lookahead bias audit

# %%
print(f"""
{'='*80}
LOOKAHEAD BIAS AUDIT
{'='*80}

iv_catm, iv_patm, iv_potm, skew_otm, parity_vspread, pc_ratio, hvol,
moneyness-bucketed OI:
  → OptionSuite snapshots for date t, known at close of t.

iv_91d_atm, iv_30d_call25, iv_30d_put25, vol_term_structure, vol_smile:
  → vsurfd snapshots for date t (OptionMetrics records at 3:59 PM),
    known at close of t.

gex, dex, delta_dollar_volume, oi_wt_delta/gamma/vega/theta:
  → Computed from end-of-day open interest and volume for date t,
    known at close of t.

rate10, rate30, rate60:
  → Date-t implied borrow rates, known at close of t.

rv_30d:
  → Uses returns from t-1 backward (30 trading days), strictly no lookahead.

vrp_rv, vrp_hvol:
  → Difference of date-t IV and backward-looking realised vol
    (both 30-day tenor, no maturity mismatch), no lookahead.

CONCLUSION: All factors are valid predictors as of close of date t,
for predicting returns from close of t to close of t+1.
{'='*80}
""")

del panel
print("Stage 6 complete.")

Crosswalk: 2,550 rows, 229 secids, 227 permnos
Converting vol surface to permno-space...
  vol_surface: 1,008,742 rows → 3,131,589 matched (100.0% of secids mapped)
  Kept columns: ['iv_91d_atm', 'iv_30d_call25', 'iv_30d_put25', 'vol_term_structure', 'vol_smile']

Converting Greeks to permno-space...
  greeks: 1,008,362 rows → 3,130,852 matched (100.0% of secids mapped)
  Kept columns: ['gex', 'dex', 'delta_dollar_volume', 'oi_wt_delta', 'oi_wt_gamma', 'oi_wt_vega', 'oi_wt_theta', 'total_oi', 'total_volume']

Converting borrow rates to permno-space...
  borrow_rates: 683,887 rows → 2,093,920 matched (100.0% of secids mapped)
  Kept columns: ['rate10', 'rate30', 'rate60']

Loading OptionSuite base...
  Base shape: (999844, 19), 216 permnos

Loading VRP...
  VRP shape: (918215, 5)

--- Sequential Merge ---
Base:          (999844, 19)
+ VRP:         (999844, 22)
+ Vol Surface: (2986372, 27)
+ Greeks:      (12047608, 36)
+ Borrow:      (45057524, 39)

Duplicate (permno, date) rows: 4405768